# Optimization — BootstrapFewShot

BootstrapFewShot is the simplest DSPy optimizer. It:
1. Runs your module on the trainset
2. Finds examples where the model got it right (using your metric)
3. Injects those as few-shot demonstrations into the prompt automatically

No prompt rewriting — just automatic few-shot example selection.

In [5]:
import numpy
import dspy
import random
from typing import Literal
from datasets import load_dataset
from dotenv import load_dotenv
load_dotenv()

lm = dspy.LM('ollama_chat/llama3.1:8b', api_base='http://localhost:11434')
dspy.configure(lm=lm)

In [6]:
# Dataset — 6 overlapping domains to create real ambiguity
TOPIC_MAP = {
    0: "society",       # Society & Culture   ↔ overlaps with politics
    1: "science",       # Science & Mathematics  ↔ overlaps with health
    2: "health",        # Health  ↔ overlaps with science
    5: "sports",        # Sports  ↔ overlaps with entertainment
    7: "entertainment", # Entertainment & Music  ↔ overlaps with sports
    9: "politics",      # Politics & Government  ↔ overlaps with society
}

raw = load_dataset("yahoo_answers_topics", split="test")
filtered = [r for r in raw if r["topic"] in TOPIC_MAP]

examples = [
    dspy.Example(question=r["question_title"], domain=TOPIC_MAP[r["topic"]]).with_inputs("question")
    for r in filtered
]

random.seed(42)

by_domain = {d: [] for d in TOPIC_MAP.values()}
for ex in examples:
    by_domain[ex.domain].append(ex)
for d in by_domain:
    random.shuffle(by_domain[d])

# 40 train + 20 dev per domain → 240 train, 120 dev
trainset = sum([by_domain[d][:40] for d in by_domain], [])
devset   = sum([by_domain[d][40:60] for d in by_domain], [])

random.shuffle(trainset)
random.shuffle(devset)

print(f"Train: {len(trainset)} | Dev: {len(devset)}")
print("Domain split:", {d: sum(1 for e in trainset if e.domain == d) for d in by_domain})

Train: 240 | Dev: 120
Domain split: {'society': 40, 'science': 40, 'health': 40, 'sports': 40, 'entertainment': 40, 'politics': 40}


In [8]:
class DomainClassifier(dspy.Signature):
    """Classify the domain of the question."""

    question: str = dspy.InputField(desc="A user's question")
    domain: Literal["science", "health", "sports", "entertainment", "politics", "society"] = dspy.OutputField(
        desc="The domain the question belongs to"
    )


class Classifier(dspy.Module):
    def __init__(self):
        self.classify = dspy.Predict(DomainClassifier)

    def forward(self, question):
        return self.classify(question=question)


def domain_accuracy(example, prediction, trace=None):
    return float(example.domain == prediction.domain)

## Baseline — unoptimized score

In [9]:
from dspy.evaluate import Evaluate

evaluate = Evaluate(devset=devset, metric=domain_accuracy, num_threads=4, display_progress=True)

baseline = Classifier()
baseline_score = evaluate(baseline)
print(f"Baseline accuracy: {baseline_score.score:.1f}%")

Average Metric: 29.00 / 43 (67.4%):  35%|███▌      | 42/120 [00:00<00:00, 131.97it/s]

2026/09/03 13:41:27 ERROR dspy.utils.parallelizer: Error for Example({'question': 'Express as a table:{(2,3),(1,-3),(10,11)?', 'domain': 'science'}) (input_keys={'question'}): 'math' is not one of ('science', 'health', 'sports', 'entertainment', 'politics', 'society'). Set `provide_traceback=True` for traceback.


Average Metric: 84.00 / 119 (70.6%): 100%|██████████| 120/120 [00:00<00:00, 372.92it/s]

2026/09/03 13:41:27 INFO dspy.evaluate.evaluate: Average Metric: 84.0 / 120 (70.0%)



Baseline accuracy: 70.0%


## Optimize with BootstrapFewShot

`max_bootstrapped_demos` — how many successful examples to inject into the prompt  
`max_labeled_demos` — how many ground-truth examples to also include

In [10]:
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=domain_accuracy,
    max_bootstrapped_demos=20,
    max_labeled_demos=10,
)

optimized = optimizer.compile(Classifier(), trainset=trainset)
print("Optimization complete.")

 10%|█         | 25/240 [00:00<00:01, 119.83it/s]

Bootstrapped 20 full traces after 25 examples for up to 1 rounds, amounting to 25 attempts.
Optimization complete.


## Compare baseline vs optimized

In [11]:
optimized_score = evaluate(optimized)

print(f"Baseline : {baseline_score.score:.1f}%")
print(f"Optimized: {optimized_score.score:.1f}%")
print(f"Delta    : +{optimized_score.score - baseline_score.score:.1f}%")

Average Metric: 89.00 / 120 (74.2%): 100%|██████████| 120/120 [00:00<00:00, 156.52it/s]

2026/09/03 13:50:00 INFO dspy.evaluate.evaluate: Average Metric: 89.0 / 120 (74.2%)



Baseline : 70.0%
Optimized: 74.2%
Delta    : +4.2%


## Inspect what changed

See the few-shot examples that were injected into the prompt.

In [12]:
optimized(question="Who won the 2022 FIFA World Cup?")
dspy.inspect_history(n=1)





[2026-09-03T13:50:03.826929]

System message:

Your input fields are:
1. `question` (str): A user's question
Your output fields are:
1. `domain` (Literal['science', 'health', 'sports', 'entertainment', 'politics', 'society']): The domain the question belongs to
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## domain ## ]]
{domain}        # note: the value you produce must exactly match (no extra characters) one of: science; health; sports; entertainment; politics; society

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the domain of the question.


User message:

[[ ## question ## ]]
define chron's disease and intestinal obstruction.?


Assistant message:

[[ ## domain ## ]]
health

[[ ## completed ## ]]


User message:

[[ ## question ## ]]
How quickly can heat transfer?


Assistant message:

[[ ## domain ## ]]
science

[[ ## completed ## ]]


User mes